In [ ]:
"""
Validação Cruzada para NER com LLMs
=====================================
Modelos:
  - Qwen3.5-9B      : unsloth/Qwen3.5-9B         → LoRA bf16
  - Gemma 4 26B A4B : unsloth/gemma-4-26B-A4B-it  → QLoRA 4-bit via FastModel (MoE)

Hiperparâmetros definidos:
  - Épocas         : 3
  - Learning Rate  : 1e-4
  - Batch Size     : 1 por GPU (gradient_accumulation_steps=4 → batch efetivo=4)
  - Quantização    : NF4 + double quantization (load_in_4bit=True onde aplicável)
  - LoRA r         : 16
  - LoRA alpha     : 64   (escala efetiva = alpha/r = 4.0)
  - LoRA dropout   : 0.05
  - target_modules : apenas atenção — q_proj, k_proj, v_proj, o_proj

Métricas de avaliação (dois níveis):
  1. seqeval IOB2   : padrão da literatura NER; P/R/F1 por rótulo + macro/micro avg;
                      penaliza spans incompletos (B-X sem I-X ≠ correto)
  2. entity-level   : compara (text, label) exatos; texto errado = FP;
                      inclui sentence accuracy

  MultiLabelBinarizer (sklearn) foi removido: é redundante com seqeval e
  ignora o texto da entidade, tornando-o o menos informativo dos três.

Diferenças de API Unsloth por modelo:
  • Qwen3.5  → FastLanguageModel  (modelos densos)
  • Gemma 4  → FastModel          (API unificada para MoE/multimodal)

Thinking mode:
  • Qwen3.5 : /no_think no prompt + remoção de <think>...</think> no parse
  • Gemma 4 : sem token <|think|> no system prompt → thinking desabilitado por padrão;
              blocos residuais removidos via regex no parse

Fold 7 reservado para hiperparâmetros do BERT — nunca usado aqui.

Correções aplicadas:
  1. dataset_text_field corrigido; datasets de treino/teste separados por responsabilidade.
  2. MLB/sklearn removidos — seqeval já cobre por rótulo + macro/micro avg.
  3. Checkpoint robusto: JSON salvo antes de avançar o contador.
  4. LoRA aplicado apenas aos módulos de atenção conforme especificado.
  5. BitsAndBytesConfig explícito: NF4 + double quantization.
  6. Acesso seguro a campos string do HuggingFace Dataset durante inferência.
  7. Campo true_labels morto removido de prepare_test_dataset.
  8. entities_to_iob2: spans ordenados por comprimento (maior tem prioridade);
     colisões detectadas e reportadas em vez de sobrescrita silenciosa.
  9. warmup_steps dinâmico: evita warmup excessivo em folds pequenos.
 10. load_in_4bit removido do load_kwargs do Gemma — BitsAndBytesConfig já o define;
     passá-lo junto causava conflito com quantization_config.
 11. Dispositivo de inferência via next(model.parameters()).device — seguro com PEFT.
 12. load_data com FileNotFoundError explícito identificando qual fold falhou."""

import os
import gc
import re
import json
import torch
from tqdm import tqdm
from datasets import Dataset
from seqeval.metrics import (
    classification_report as seq_classification_report,
    precision_score,
    recall_score,
    f1_score,
)
from transformers import BitsAndBytesConfig
from trl import SFTTrainer, SFTConfig


# ============================================================
# CONFIGURAÇÕES — altere apenas MODEL_CHOICE
# ============================================================

# "qwen"  → Qwen3.5-9B       (LoRA bf16,    ~18 GB treino)
# "gemma" → Gemma 4 26B A4B  (QLoRA 4-bit,  ~22 GB treino)
MODEL_CHOICE = "qwen"

# BitsAndBytesConfig explícito: NF4 + dupla quantização
NF4_CONFIG = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

MODEL_REGISTRY = {
    "qwen": {
        "hf_id":             "unsloth/Qwen3.5-9B",
        # QLoRA 4-bit NÃO recomendado para Qwen3.5 → LoRA bf16 puro
        "load_in_4bit":      False,
        "quantization_config": None,
        "dtype":             torch.bfloat16,
        "api":               "fast_language",
    },
    "gemma": {
        "hf_id":             "unsloth/gemma-4-26B-A4B-it",
        # QLoRA 4-bit necessário para caber em 24 GB
        "load_in_4bit":      True,
        "quantization_config": NF4_CONFIG,
        "dtype":             None,          # auto-detecção recomendada pelo Unsloth
        "api":               "fast_model",
    },
}

CFG        = MODEL_REGISTRY[MODEL_CHOICE]
MODEL_NAME = CFG["hf_id"]

# ---- Hiperparâmetros ----
MAX_LEN    = 2048
MAX_NEW    = 512
NUM_EPOCHS = 3
LR         = 1e-4
BATCH_SIZE = 1          # por GPU
GRAD_ACCUM = 4          # batch efetivo = BATCH_SIZE * GRAD_ACCUM = 4

# ---- LoRA ----
LORA_R       = 16
LORA_ALPHA   = 64       # escala = alpha / r = 4.0
LORA_DROPOUT = 0.05
# Apenas módulos de atenção conforme especificado
LORA_TARGETS = ["q_proj", "k_proj", "v_proj", "o_proj"]

BASE_DIR = "../Partitions/Datasets/Paramopama/divisions/"
LABELS   = ["LOCAL", "ORGANIZACAO", "PESSOA", "TEMPO"]

# Fold reservado para ajuste de hiperparâmetros do BERT — NUNCA usado aqui
HPARAM_FOLD = 7
ALL_FOLDS   = [i for i in range(10) if i != HPARAM_FOLD]   # [0,1,2,3,4,5,6,8,9]

CHECKPOINT_FILE = "ultimo_fold.txt"

os.makedirs("modelos_por_fold", exist_ok=True)
open("relatorio_folds.txt", "a", encoding="utf-8").close()


# ============================================================
# PROMPT
# ============================================================

def build_instruction() -> str:
    """
    Qwen3.5 : /no_think desabilita raciocínio estendido.
    Gemma 4 : thinking desabilitado por padrão sem <|think|> no system prompt.
    """
    base = (
        "Extraia entidades nomeadas do texto abaixo e retorne "
        "SOMENTE um array JSON com objetos contendo os campos "
        "'text' e 'label'. Se não houver entidades, retorne [].\n"
        f"Rótulos válidos: {', '.join(LABELS)}."
    )
    return ("/no_think\n" + base) if MODEL_CHOICE == "qwen" else base


INSTRUCTION = build_instruction()


# ============================================================
# FUNÇÕES AUXILIARES
# ============================================================

def load_data(path: str, col_sep: str = "\t",
              token_col: int = 0, tag_col: int = -1) -> list:
    """
    Carrega um arquivo de divisao em formato JSON ou CoNLL.

    Parametros
    ----------
    path      : caminho do arquivo
    col_sep   : separador de colunas — "\t" (HAREM) ou " " (leNER, UlyssesNER-BR)
    token_col : indice da coluna do token  (padrao: 0 — primeira coluna)
    tag_col   : indice da coluna da tag    (padrao: -1 — ultima coluna)
                Usar -1 e seguro para qualquer numero de colunas:
                  2 colunas  → [token, tag]          → tag_col=-1 pega tag  ✅
                  4 colunas  → [token,lemma,pos,tag]  → tag_col=-1 pega tag  ✅

    Datasets suportados
    -------------------
    HAREM        : col_sep="\t", token_col=0, tag_col=-1  (2 colunas)
    leNER        : col_sep=" ",  token_col=0, tag_col=-1  (2 colunas)
    UlyssesNER-BR: col_sep=" ",  token_col=0, tag_col=-1  (2 colunas)
    CoNLL-2003   : col_sep=" ",  token_col=0, tag_col=-1  (4 colunas — pega ultima)

    JSON esperado:
        [{"text": "...", "entities": [{"text": "...", "label": "..."}]}, ...]
    """
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"[Erro] Arquivo nao encontrado: '{path}'\n"
            f"Verifique se BASE_DIR esta correto: '{BASE_DIR}'"
        )

    ext = os.path.splitext(path)[1].lower()
    if ext == ".json":
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)

    return _load_conll(path, col_sep=col_sep, token_col=token_col, tag_col=tag_col)


def _load_conll(path: str, col_sep: str = "\t",
                token_col: int = 0, tag_col: int = -1) -> list:
    """
    Parser CoNLL -> lista de dicts {"text": str, "entities": list}.

    Separador de sentença: token '.' com tag 'O' — o ponto final é incluído
    no texto da sentença e depois a sentença é fechada.
    Linhas em branco são ignoradas — não afetam a segmentação.

    Suporta qualquer numero de colunas — token_col e tag_col definem
    quais colunas usar, independente do total de colunas na linha.

    tag_col=-1 significa ultima coluna, o que e correto para:
      - 2 colunas [token, tag]
      - 4 colunas [token, lemma, pos, tag]  (CoNLL-2003 e similares)

    Logica IOB2:
      B-X           → abre novo span
      I-X (mesmo)   → continua span
      I-X (diferente ou sem B anterior) → loga aviso, abre novo span
      O / fim sent. → fecha span pendente
    """
    samples  = []
    tokens   = []
    bio_tags = []

    def close_span(span_tokens, span_label, entities):
        if span_tokens and span_label and span_label in LABELS:
            entities.append({
                "text":  " ".join(span_tokens),
                "label": span_label,
            })

    def flush_sentence(tokens, bio_tags):
        if not tokens:
            return
        text      = " ".join(tokens)
        entities  = []
        span_toks = []
        span_lbl  = None

        for token, tag in zip(tokens, bio_tags):
            if tag.startswith("B-"):
                close_span(span_toks, span_lbl, entities)
                span_toks = [token]
                span_lbl  = tag[2:]
            elif tag.startswith("I-"):
                label = tag[2:]
                if span_lbl == label:
                    span_toks.append(token)
                else:
                    print(
                        f"[CoNLL] I- sem B- anterior ('{tag}' apos '{span_lbl}')"
                        f" em: \"{text[:80]}\""
                    )
                    close_span(span_toks, span_lbl, entities)
                    span_toks = [token]
                    span_lbl  = label
            else:
                close_span(span_toks, span_lbl, entities)
                span_toks = []
                span_lbl  = None

        close_span(span_toks, span_lbl, entities)
        samples.append({"text": text, "entities": entities})

    with open(path, "r", encoding="utf-8") as f:
        # Tokens que encerram sentença quando rotulados como 'O'
        SENTENCE_ENDINGS = {".", "?", "!", "...", "…"}

        for raw_line in f:
            line = raw_line.rstrip("\n")

            # Linhas em branco ignoradas — separação feita pelos tokens de fechamento
            if line.strip() == "":
                continue

            parts = line.split(col_sep)

            if len(parts) < 2:
                print(f"[CoNLL] Linha malformada ignorada: {line!r}")
                continue

            try:
                token = parts[token_col]
                tag   = parts[tag_col].strip()
            except IndexError:
                print(f"[CoNLL] Coluna ausente (token_col={token_col}, tag_col={tag_col}): {line!r}")
                continue

            tokens.append(token)
            bio_tags.append(tag)

            # Token de fechamento rotulado como O → encerra a sentença
            if token in SENTENCE_ENDINGS and tag == "O":
                flush_sentence(tokens, bio_tags)
                tokens   = []
                bio_tags = []

    # Fecha sentença residual (arquivo sem ponto final na última sentença)
    flush_sentence(tokens, bio_tags)

    # ---- Debug: amostra das primeiras 3 sentenças lidas ----
    print(f"\n[CoNLL] {path}: {len(samples)} sentenças carregadas.")
    print("[CoNLL] Amostra das primeiras 3 sentenças lidas:")
    for i, s in enumerate(samples[:3]):
        print(f"  Sentença {i+1}:")
        print(f"    text     : {s['text'][:80]!r}{'...' if len(s['text']) > 80 else ''}")
        print(f"    entities : {s['entities']}")
    sem_entidade = sum(1 for s in samples if not s["entities"])
    print(f"[CoNLL] Sentenças sem entidade: {sem_entidade}/{len(samples)} "
          f"({100*sem_entidade/len(samples):.1f}%)\n")

    return samples
def prepare_train_dataset(data: list) -> Dataset:
    """
    Dataset de TREINO: contém apenas os campos necessários para o SFTTrainer.
    Sentenças sem entidade geram output '[]' — incluídas normalmente.
    """
    samples = []
    for item in data:
        ents = item.get("entities", [])
        ents_fmt = [
            {"text": e["text"], "label": e["label"]}
            for e in ents if e["label"] in LABELS
        ]
        prompt = (
            f"{INSTRUCTION}\n"
            f"Texto: {item['text']}\n"
            f"Entidades: {json.dumps(ents_fmt, ensure_ascii=False)}"
        )
        samples.append({"text": prompt})

    # ---- Debug: amostra dos prompts de treino ----
    print(f"\n[Treino] {len(samples)} prompts montados.")
    print("[Treino] Amostra de 2 prompts — 1 com entidade, 1 sem:")
    com_ent = next((s for s in samples if "label" in s["text"]), None)
    sem_ent = next((s for s in samples if '"entities": []' in s["text"]
                    or s["text"].endswith("[]")), None)
    for label, sample in [("COM entidade", com_ent), ("SEM entidade", sem_ent)]:
        if sample:
            print(f"\n  [{label}]")
            for linha in sample["text"].splitlines():
                print(f"    {linha[:100]}")

    return Dataset.from_list(samples)


def prepare_test_dataset(data: list) -> Dataset:
    """
    Dataset de TESTE: contém metadados necessários para avaliação.
    Campos string armazenados diretamente (sem tokenização).
    """
    samples = []
    for item in data:
        ents = item.get("entities", [])
        ents_fmt = [
            {"text": e["text"], "label": e["label"]}
            for e in ents if e["label"] in LABELS
        ]
        samples.append({
            "input_text":    item["text"],
            "true_entities": json.dumps(ents_fmt, ensure_ascii=False),
            # true_labels removido — nunca usado; rótulos já estão dentro de true_entities
        })

    # ---- Debug: amostra do dataset de teste ----
    print(f"\n[Teste] {len(samples)} sentenças no dataset de teste.")
    print("[Teste] Amostra das primeiras 3 entradas:")
    for i, s in enumerate(samples[:3]):
        ents = json.loads(s["true_entities"])
        print(f"  Entrada {i+1}:")
        print(f"    input_text    : {s['input_text'][:80]!r}"
              f"{'...' if len(s['input_text']) > 80 else ''}")
        print(f"    true_entities : {ents}")
    sem_ent = sum(1 for s in samples if s["true_entities"] == "[]")
    print(f"[Teste] Sentenças sem entidade: {sem_ent}/{len(samples)} "
          f"({100*sem_ent/len(samples):.1f}%)\n")

    return Dataset.from_list(samples)


def entities_to_iob2(text: str, entities: list) -> list:
    """
    Converte lista de entidades para sequência IOB2 por token (split por espaço).
    seqeval penaliza entidades incompletas: B-X sem I-X ≠ span completo.

    Colisão de spans: se dois spans compartilham tokens (substring vs. string maior),
    o span mais longo tem prioridade — o menor é ignorado com aviso, evitando
    sobrescrita silenciosa de tags que corromperia a sequência IOB2.
    """
    tokens     = text.split()
    tags       = ["O"] * len(tokens)
    # Ordena por comprimento decrescente para dar prioridade a spans maiores
    sorted_ents = sorted(entities, key=lambda e: len(e["text"].split()), reverse=True)

    for ent in sorted_ents:
        ent_tokens = ent["text"].split()
        label      = ent["label"]
        n          = len(ent_tokens)
        matched    = False
        for i in range(len(tokens) - n + 1):
            if tokens[i : i + n] == ent_tokens:
                # Verifica colisão: algum token do span já foi marcado?
                span_tags = tags[i : i + n]
                if any(t != "O" for t in span_tags):
                    print(
                        f"[IOB2] Colisão ignorada: '{ent['text']}' "
                        f"sobrepõe span já marcado em: \"{text[:80]}...\""
                    )
                    matched = True
                    break
                tags[i] = f"B-{label}"
                for j in range(1, n):
                    tags[i + j] = f"I-{label}"
                matched = True
                break
        if not matched:
            # Entidade não encontrada por split simples (pode conter pontuação colada)
            print(f"[IOB2] Entidade não alinhada ignorada: '{ent['text']}'")
    return tags


def parse_model_output(output_text: str) -> list:
    """
    Extrai JSON de entidades da saída do modelo.
    Remove blocos de thinking de ambos os modelos:
      - Qwen3.5 : <think>...</think>
      - Gemma 4 : <thought>...</thought>
    Loga avisos em vez de silenciar erros — permite identificar problemas
    sistemáticos de formato na saída do modelo durante a avaliação.
    """
    output_text = re.sub(r"<think>.*?</think>",     "", output_text, flags=re.DOTALL)
    output_text = re.sub(r"<thought>.*?</thought>", "", output_text, flags=re.DOTALL)
    output_text = output_text.strip()

    start = output_text.find("[")
    end   = output_text.rfind("]")

    if start == -1 or end == -1:
        if output_text:  # silencia apenas saídas genuinamente vazias
            print(f"[Parse] Nenhum array JSON encontrado. Saída bruta: {output_text[:120]!r}")
        return []

    try:
        parsed = json.loads(output_text[start : end + 1])
    except json.JSONDecodeError as e:
        print(f"[Parse] JSONDecodeError: {e} | Saída bruta: {output_text[:120]!r}")
        return []

    result = []
    for e in parsed:
        if not isinstance(e, dict):
            print(f"[Parse] Elemento ignorado (não é dict): {e!r}")
            continue
        if "text" not in e:
            print(f"[Parse] Elemento sem campo 'text' ignorado: {e!r}")
            continue
        if e.get("label") not in LABELS:
            print(f"[Parse] Rótulo inválido ignorado: {e.get('label')!r} em {e!r}")
            continue
        result.append({"text": e["text"], "label": e["label"]})

    return result


# ============================================================
# CHECKPOINT  (robusto: JSON salvo ANTES de avançar o contador)
# ============================================================

def read_last_fold() -> int:
    if os.path.exists(CHECKPOINT_FILE):
        with open(CHECKPOINT_FILE, "r") as f:
            try:
                return int(f.read().strip())
            except ValueError:
                return 0
    return 0


def write_last_fold(loop_idx: int):
    with open(CHECKPOINT_FILE, "w") as f:
        f.write(str(loop_idx))


# ============================================================
# CARREGAMENTO DO MODELO VIA UNSLOTH
# ============================================================

def load_model_and_tokenizer():
    """
    Qwen3.5-9B  → FastLanguageModel (API para modelos densos)
      • LoRA bf16 puro — QLoRA 4-bit não recomendado para Qwen3.5
      • VRAM estimada: ~18 GB

    Gemma 4 26B A4B → FastModel (API unificada Unsloth para MoE/multimodal)
      • QLoRA 4-bit: NF4 + double quantization
      • dtype=None: auto-detecção recomendada pelo Unsloth
      • VRAM estimada: ~22 GB

    LoRA aplicado apenas aos módulos de atenção (q_proj, k_proj, v_proj, o_proj).
    """
    load_kwargs = dict(
        model_name    = MODEL_NAME,
        max_seq_length= MAX_LEN,
        dtype         = CFG["dtype"],
    )
    if CFG["quantization_config"] is not None:
        # Gemma: quantization_config (BitsAndBytesConfig) já define load_in_4bit=True
        # internamente — passar load_in_4bit separado causaria conflito/aviso
        load_kwargs["quantization_config"] = CFG["quantization_config"]
    else:
        # Qwen: sem quantização 4-bit, passa load_in_4bit=False explicitamente
        load_kwargs["load_in_4bit"] = CFG["load_in_4bit"]

    lora_kwargs = dict(
        r                          = LORA_R,
        lora_alpha                 = LORA_ALPHA,
        lora_dropout               = LORA_DROPOUT,
        target_modules             = LORA_TARGETS,
        bias                       = "none",
        use_gradient_checkpointing = "unsloth",
        random_state               = 42,
    )

    if CFG["api"] == "fast_language":
        from unsloth import FastLanguageModel
        model, tokenizer = FastLanguageModel.from_pretrained(**load_kwargs)
        model = FastLanguageModel.get_peft_model(model, **lora_kwargs)
        model._unsloth_api = "fast_language"
    else:
        from unsloth import FastModel
        model, tokenizer = FastModel.from_pretrained(**load_kwargs)
        model = FastModel.get_peft_model(model, **lora_kwargs)
        model._unsloth_api = "fast_model"

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"

    return model, tokenizer


def set_inference_mode(model):
    """Ativa modo de inferência otimizado do Unsloth conforme a API usada."""
    if getattr(model, "_unsloth_api", "fast_language") == "fast_model":
        from unsloth import FastModel
        FastModel.for_inference(model)
    else:
        from unsloth import FastLanguageModel
        FastLanguageModel.for_inference(model)


# ============================================================
# AVALIAÇÃO
# ============================================================

def evaluate_fold(fold_idx: int, model, tokenizer, ds_test: Dataset) -> tuple:
    """
    Dois níveis de avaliação (MLB/sklearn removido — redundante e menos rigoroso):
      1. entity-level : compara (text, label) exatos — texto errado = FP
      2. seqeval IOB2 : padrão da literatura NER; penaliza spans incompletos
                        e já fornece P/R/F1 por rótulo + micro/macro avg

    Sentenças sem entidade (true=∅, pred=∅) contam como corretas.
    Campos do dataset de teste acessados via str() — seguro para HuggingFace Dataset.
    """
    set_inference_mode(model)

    total_tp = total_fp = total_fn = 0
    correct_sentences = total_sentences = 0

    seqeval_true: list = []
    seqeval_pred: list = []
    errors:       list = []

    for item in tqdm(ds_test, desc=f"Avaliando Fold {fold_idx}"):
        # Acesso seguro: campos vêm como str do Dataset HuggingFace
        input_text    = str(item["input_text"])
        true_entities = json.loads(str(item["true_entities"]))

        prompt = f"{INSTRUCTION}\nTexto: {input_text}\nEntidades:"

        inputs = tokenizer(
            prompt,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_LEN,
        ).to(next(model.parameters()).device)  # seguro com PEFT + Unsloth

        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW,
                use_cache=True,
            )

        # Decodifica apenas tokens novos (descarta o prompt)
        new_ids     = output_ids[0][inputs["input_ids"].shape[1]:]
        output_text = tokenizer.decode(new_ids, skip_special_tokens=True)

        pred_entities = parse_model_output(output_text)

        # ---- entity-level (text + label) ----
        true_set = {(e["text"], e["label"]) for e in true_entities}
        pred_set = {(e["text"], e["label"]) for e in pred_entities}

        total_tp += len(true_set & pred_set)
        total_fp += len(pred_set - true_set)
        total_fn += len(true_set - pred_set)
        total_sentences += 1
        if true_set == pred_set:
            correct_sentences += 1

        # ---- seqeval IOB2 ----
        seqeval_true.append(entities_to_iob2(input_text, true_entities))
        seqeval_pred.append(entities_to_iob2(input_text, pred_entities))

        # ---- registro de erros ----
        if true_set != pred_set:
            errors.append({
                "text":          input_text,
                "true_entities": true_entities,
                "pred_entities": pred_entities,
                "missed":        list(true_set - pred_set),
                "spurious":      list(pred_set - true_set),
            })

    # ---- métricas entity-level ----
    p_ent  = total_tp / (total_tp + total_fp) if (total_tp + total_fp) > 0 else 0.0
    r_ent  = total_tp / (total_tp + total_fn) if (total_tp + total_fn) > 0 else 0.0
    f1_ent = (2 * p_ent * r_ent / (p_ent + r_ent)) if (p_ent + r_ent) > 0 else 0.0
    ac_ent = correct_sentences / total_sentences if total_sentences > 0 else 0.0

    # ---- seqeval: micro global + per-label ----
    # seq_classification_report já contém macro avg, micro avg e por rótulo
    seq_report = seq_classification_report(
        seqeval_true, seqeval_pred, output_dict=True, zero_division=0
    )
    seq_p  = precision_score(seqeval_true, seqeval_pred, zero_division=0)
    seq_r  = recall_score(seqeval_true,   seqeval_pred, zero_division=0)
    seq_f1 = f1_score(seqeval_true,       seqeval_pred, zero_division=0)

    metrics = {
        "entity_level": {
            "precision": p_ent,
            "recall":    r_ent,
            "f1":        f1_ent,
            "accuracy":  ac_ent,
        },
        "seqeval": {
            "precision": seq_p,
            "recall":    seq_r,
            "f1":        seq_f1,
            "per_label": seq_report,   # contém por rótulo + macro/micro avg
        },
    }
    return metrics, errors


# ============================================================
# MAIN
# ============================================================

def main():
    # ----------------------------------------------------------------
    # Configuração do dataset — altere aqui ao trocar de corpus
    # ----------------------------------------------------------------
    # HAREM        : col_sep="\t", file_ext=".txt"  (2 colunas: token tag)
    # leNER        : col_sep=" ",  file_ext=".conll" (2 colunas: token tag)
    # UlyssesNER-BR: col_sep=" ",  file_ext=".conll" (2 colunas: token tag)
    # CoNLL-2003   : col_sep=" ",  file_ext=".conll" (4 colunas: token lemma pos tag)
    #                → tag_col=-1 pega sempre a última coluna, correto para todos
    DATASET_COL_SEP  = "\t"    # separador de colunas do arquivo CoNLL
    DATASET_FILE_EXT = ".txt"  # extensão dos arquivos de divisão
    DATASET_TOKEN_COL = 0      # índice da coluna do token
    DATASET_TAG_COL   = -1     # índice da coluna da tag (-1 = última)

    def _load(path: str) -> list:
        """Wrapper que injeta os parâmetros do dataset em load_data."""
        return load_data(
            path,
            col_sep   = DATASET_COL_SEP,
            token_col = DATASET_TOKEN_COL,
            tag_col   = DATASET_TAG_COL,
        )

    division_files = {i: f"division_{i}{DATASET_FILE_EXT}" for i in ALL_FOLDS}
    all_reports    = []
    start_loop_idx = read_last_fold()

    for loop_idx, fold in enumerate(ALL_FOLDS):

        # Fold já processado em execução anterior
        if loop_idx < start_loop_idx:
            result_path = f"resultados_fold_{fold}.json"
            if os.path.exists(result_path):
                print(f"[Checkpoint] Fold {fold} já processado. Carregando...")
                with open(result_path, "r", encoding="utf-8") as f:
                    all_reports.append(json.load(f))
                continue   # só pula se o arquivo realmente existe
            else:
                # Arquivo ausente: reseta o checkpoint e reprocessa este fold
                print(
                    f"[Aviso] Fold {fold} marcado como processado, "
                    "mas arquivo de resultados não encontrado. Reprocessando..."
                )
                start_loop_idx = loop_idx
                # NÃO dá continue — cai no fluxo normal abaixo

        print(f"\n{'='*55}")
        print(f"  FOLD {fold} (loop {loop_idx}) — {MODEL_NAME}")
        print(f"{'='*55}")

        # ---- Dados ----
        test_data  = _load(os.path.join(BASE_DIR, division_files[fold]))
        train_data = []
        for j, fname in division_files.items():
            if j != fold:
                train_data.extend(_load(os.path.join(BASE_DIR, fname)))

        # Datasets separados por responsabilidade
        ds_train = prepare_train_dataset(train_data)   # apenas campo "text" para o SFTTrainer
        ds_test  = prepare_test_dataset(test_data)     # metadados para avaliação

        # ---- Modelo ----
        model, tokenizer = load_model_and_tokenizer()

        # ---- Treino ----
        # Warmup dinâmico: mínimo de 10 steps ou 5% dos steps totais,
        # evitando warmup excessivo em folds com poucos exemplos.
        steps_per_epoch = max(1, len(ds_train) // (BATCH_SIZE * GRAD_ACCUM))
        total_steps     = steps_per_epoch * NUM_EPOCHS
        warmup_steps    = max(10, int(0.05 * total_steps))

        sft_cfg = SFTConfig(
            output_dir                  = f"modelos_por_fold/fold_{fold}",
            per_device_train_batch_size = BATCH_SIZE,
            gradient_accumulation_steps = GRAD_ACCUM,
            num_train_epochs            = NUM_EPOCHS,
            learning_rate               = LR,
            warmup_steps                = warmup_steps,
            logging_steps               = 10,
            bf16                        = True,
            fp16                        = False,
            save_strategy               = "no",
            eval_strategy               = "no",
            report_to                   = "none",
            push_to_hub                 = False,
            max_seq_length              = MAX_LEN,
            dataset_text_field          = "text",   # campo correto do ds_train
        )

        trainer = SFTTrainer(
            model         = model,
            tokenizer     = tokenizer,
            train_dataset = ds_train,
            args          = sft_cfg,
        )
        trainer.train()

        # ---- Avaliação ----
        metrics, errors = evaluate_fold(fold, model, tokenizer, ds_test)

        # ---- Salvar erros ----
        with open(f"erros_fold_{fold}.txt", "w", encoding="utf-8") as f_err:
            for err in errors:
                f_err.write("-" * 80 + "\n")
                f_err.write(f"Texto:\n{err['text']}\n\n")
                f_err.write("Entidades verdadeiras:\n")
                f_err.write(json.dumps(err["true_entities"], ensure_ascii=False, indent=2))
                f_err.write("\n\nEntidades previstas:\n")
                f_err.write(json.dumps(err["pred_entities"], ensure_ascii=False, indent=2))
                f_err.write(f"\n\nPerdidas  (FN): {err['missed']}")
                f_err.write(f"\nEspúrias  (FP): {err['spurious']}\n\n")

        # ---- Salvar métricas (ANTES de avançar o checkpoint) ----
        result_path = f"resultados_fold_{fold}.json"
        with open(result_path, "w", encoding="utf-8") as f:
            json.dump(metrics, f, ensure_ascii=False, indent=2)

        # ---- Relatório por fold ----
        el       = metrics["entity_level"]
        sv       = metrics["seqeval"]
        per_lbl  = sv["per_label"]                         # dict do seq_classification_report
        seq_ma   = per_lbl.get("macro avg",  {})
        seq_mi   = per_lbl.get("micro avg",  {})

        with open("relatorio_folds.txt", "a", encoding="utf-8") as f:
            f.write(f"\n====== FOLD {fold} — {MODEL_NAME} ======\n")
            f.write("  seqeval por rótulo:\n")
            for label in LABELS:
                r = per_lbl.get(label, {})
                f.write(
                    f"    {label}: "
                    f"P={r.get('precision', 0):.3f}  "
                    f"R={r.get('recall', 0):.3f}  "
                    f"F1={r.get('f1-score', 0):.3f}\n"
                )
            f.write(
                f"\n  seqeval Macro Avg : P={seq_ma.get('precision',0):.3f}  "
                f"R={seq_ma.get('recall',0):.3f}  F1={seq_ma.get('f1-score',0):.3f}\n"
                f"  seqeval Micro Avg : P={seq_mi.get('precision',0):.3f}  "
                f"R={seq_mi.get('recall',0):.3f}  F1={seq_mi.get('f1-score',0):.3f}\n"
                f"\n  Entity-level (text+label):\n"
                f"    P={el['precision']:.3f}  R={el['recall']:.3f}  "
                f"F1={el['f1']:.3f}  Acc={el['accuracy']:.3f}\n"
                + "=" * 55 + "\n"
            )

        all_reports.append(metrics)

        # Checkpoint avançado APÓS o JSON ser salvo com sucesso
        write_last_fold(loop_idx + 1)

        # ---- Liberar VRAM ----
        del model, trainer, tokenizer
        torch.cuda.empty_cache()
        gc.collect()

    # ============================================================
    # RELATÓRIO FINAL
    # ============================================================
    n = len(all_reports)
    if n == 0:
        print("Nenhum fold processado.")
        return

    def avg(key_path: list) -> float:
        """Média de uma métrica ao longo dos folds, ignorando folds sem a chave."""
        total = 0.0
        valid = 0
        for rep in all_reports:
            node = rep
            try:
                for k in key_path:
                    node = node[k]
                total += float(node)
                valid += 1
            except (KeyError, TypeError):
                pass
        return total / valid if valid > 0 else 0.0

    # Médias por rótulo via seqeval (per_label)
    final_per_label = {}
    for label in LABELS:
        final_per_label[label] = {
            "precision": avg(["seqeval", "per_label", label, "precision"]),
            "recall":    avg(["seqeval", "per_label", label, "recall"]),
            "f1":        avg(["seqeval", "per_label", label, "f1-score"]),
        }

    seq_macro = {
        k: avg(["seqeval", "per_label", "macro avg", k])
        for k in ("precision", "recall", "f1-score")
    }
    seq_micro = {
        k: avg(["seqeval", "per_label", "micro avg", k])
        for k in ("precision", "recall", "f1-score")
    }
    entity_avg = {
        k: avg(["entity_level", k])
        for k in ("precision", "recall", "f1", "accuracy")
    }

    report_lines = [
        f"=== MÉDIAS DOS {n} FOLDS ===",
        f"Modelo : {MODEL_NAME}",
        f"(Fold {HPARAM_FOLD} excluído — reservado para hiperparâmetros do BERT)",
        f"Hiperparâmetros: épocas={NUM_EPOCHS} | lr={LR} | batch={BATCH_SIZE} "
        f"| grad_accum={GRAD_ACCUM} | r={LORA_R} | alpha={LORA_ALPHA} | dropout={LORA_DROPOUT}",
        "",
        "  seqeval por rótulo (média dos folds):",
    ]
    for label, s in final_per_label.items():
        report_lines.append(
            f"    {label}: P={s['precision']:.3f}  R={s['recall']:.3f}  F1={s['f1']:.3f}"
        )
    report_lines += [
        "",
        f"  seqeval Macro Avg : P={seq_macro['precision']:.3f}  "
        f"R={seq_macro['recall']:.3f}  F1={seq_macro['f1-score']:.3f}",
        f"  seqeval Micro Avg : P={seq_micro['precision']:.3f}  "
        f"R={seq_micro['recall']:.3f}  F1={seq_micro['f1-score']:.3f}",
        "",
        "  Entity-level (text+label exatos):",
        f"    P={entity_avg['precision']:.3f}  R={entity_avg['recall']:.3f}  "
        f"F1={entity_avg['f1']:.3f}  Acc={entity_avg['accuracy']:.3f}",
    ]

    report_text = "\n".join(report_lines) + "\n"

    with open("relatorio_final.txt", "w", encoding="utf-8") as f:
        f.write(report_text)

    print(report_text)
    print(f"✅ Concluído. Modelo: {MODEL_NAME} | Folds processados: {n}")


if __name__ == "__main__":
    main()

ModuleNotFoundError: No module named 'torch'